# Common Data Quality Rules

**Purpose:** Reusable data quality validation rules and helpers.

**Usage:**
```python
%run "./common/common_dq_rules"

# Validate DataFrame
results = validate_not_null(df, ["id", "name", "email"])
results = validate_unique(df, ["id"])
results = validate_format(df, "email", r"^[\w.-]+@[\w.-]+\.\w+$")

# Get validation summary
summary = get_validation_summary(df, validation_rules)
```

**Features:**
* Null checks
* Uniqueness validation
* Format/regex validation
* Range checks
* Referential integrity
* Custom rule support

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, isnull, lit, regexp_extract, sum as spark_sum
from typing import List, Dict, Any, Optional, Callable
from dataclasses import dataclass

# ============================================================================
# DATA QUALITY RULES
# ============================================================================

@dataclass
class ValidationResult:
    """Result of a data quality validation."""
    rule_name: str
    passed: bool
    total_rows: int
    failed_rows: int
    pass_rate: float
    message: str

In [0]:
def validate_not_null(
    df: DataFrame,
    columns: List[str],
    rule_name: str = "not_null"
) -> List[ValidationResult]:
    """
    Validate that columns do not contain null values.
    
    Args:
        df: Input DataFrame
        columns: Columns to check
        rule_name: Name for this validation rule
    
    Returns:
        List of ValidationResult objects
    
    Example:
        results = validate_not_null(df, ["id", "name", "email"])
        for result in results:
            if not result.passed:
                print(f"Failed: {result.message}")
    """
    total_rows = df.count()
    results = []
    
    for column in columns:
        null_count = df.filter(col(column).isNull()).count()
        passed = (null_count == 0)
        pass_rate = 1.0 - (null_count / total_rows) if total_rows > 0 else 1.0
        
        results.append(ValidationResult(
            rule_name=f"{rule_name}_{column}",
            passed=passed,
            total_rows=total_rows,
            failed_rows=null_count,
            pass_rate=pass_rate,
            message=f"Column '{column}': {null_count:,} null values found"
        ))
    
    return results

def validate_unique(
    df: DataFrame,
    columns: List[str],
    rule_name: str = "unique"
) -> ValidationResult:
    """
    Validate that column combination is unique.
    
    Args:
        df: Input DataFrame
        columns: Columns to check for uniqueness
        rule_name: Name for this validation rule
    
    Returns:
        ValidationResult object
    
    Example:
        result = validate_unique(df, ["id"])
        result = validate_unique(df, ["order_id", "product_id"])
    """
    total_rows = df.count()
    distinct_count = df.select(columns).distinct().count()
    duplicate_count = total_rows - distinct_count
    passed = (duplicate_count == 0)
    pass_rate = distinct_count / total_rows if total_rows > 0 else 1.0
    
    columns_str = ", ".join(columns)
    
    return ValidationResult(
        rule_name=f"{rule_name}_{columns_str}",
        passed=passed,
        total_rows=total_rows,
        failed_rows=duplicate_count,
        pass_rate=pass_rate,
        message=f"Columns [{columns_str}]: {duplicate_count:,} duplicate records found"
    )

def validate_format(
    df: DataFrame,
    column: str,
    pattern: str,
    rule_name: Optional[str] = None
) -> ValidationResult:
    """
    Validate column values match a regex pattern.
    
    Args:
        df: Input DataFrame
        column: Column to validate
        pattern: Regex pattern to match
        rule_name: Name for this validation rule
    
    Returns:
        ValidationResult object
    
    Example:
        # Email format
        result = validate_format(df, "email", r"^[\w.-]+@[\w.-]+\.\w+$")
        
        # Phone format
        result = validate_format(df, "phone", r"^\d{3}-\d{3}-\d{4}$")
    """
    if rule_name is None:
        rule_name = f"format_{column}"
    
    total_rows = df.count()
    
    # Count rows that don't match pattern (excluding nulls)
    invalid_count = df.filter(
        col(column).isNotNull() &
        (regexp_extract(col(column), pattern, 0) == "")
    ).count()
    
    passed = (invalid_count == 0)
    pass_rate = 1.0 - (invalid_count / total_rows) if total_rows > 0 else 1.0
    
    return ValidationResult(
        rule_name=rule_name,
        passed=passed,
        total_rows=total_rows,
        failed_rows=invalid_count,
        pass_rate=pass_rate,
        message=f"Column '{column}': {invalid_count:,} values do not match pattern"
    )

In [0]:
def validate_range(
    df: DataFrame,
    column: str,
    min_value: Optional[float] = None,
    max_value: Optional[float] = None,
    rule_name: Optional[str] = None
) -> ValidationResult:
    """
    Validate numeric column is within range.
    
    Args:
        df: Input DataFrame
        column: Column to validate
        min_value: Minimum allowed value (inclusive)
        max_value: Maximum allowed value (inclusive)
        rule_name: Name for this validation rule
    
    Returns:
        ValidationResult object
    
    Example:
        # Age between 0 and 120
        result = validate_range(df, "age", min_value=0, max_value=120)
        
        # Price must be positive
        result = validate_range(df, "price", min_value=0)
    """
    if rule_name is None:
        rule_name = f"range_{column}"
    
    total_rows = df.count()
    
    # Build filter condition
    condition = col(column).isNotNull()
    
    if min_value is not None:
        condition = condition & (col(column) < min_value)
    
    if max_value is not None:
        condition = condition & (col(column) > max_value)
    
    out_of_range_count = df.filter(condition).count()
    passed = (out_of_range_count == 0)
    pass_rate = 1.0 - (out_of_range_count / total_rows) if total_rows > 0 else 1.0
    
    range_str = f"[{min_value}, {max_value}]"
    
    return ValidationResult(
        rule_name=rule_name,
        passed=passed,
        total_rows=total_rows,
        failed_rows=out_of_range_count,
        pass_rate=pass_rate,
        message=f"Column '{column}': {out_of_range_count:,} values outside range {range_str}"
    )

def validate_referential_integrity(
    child_df: DataFrame,
    parent_df: DataFrame,
    child_key: str,
    parent_key: str,
    rule_name: str = "referential_integrity"
) -> ValidationResult:
    """
    Validate foreign key relationship (referential integrity).
    
    Args:
        child_df: Child DataFrame
        parent_df: Parent DataFrame
        child_key: Foreign key column in child
        parent_key: Primary key column in parent
        rule_name: Name for this validation rule
    
    Returns:
        ValidationResult object
    
    Example:
        # Validate orders.customer_id references customers.id
        result = validate_referential_integrity(
            child_df=orders_df,
            parent_df=customers_df,
            child_key="customer_id",
            parent_key="id"
        )
    """
    total_rows = child_df.count()
    
    # Find orphaned records (child keys not in parent)
    orphaned_df = child_df.join(
        parent_df.select(col(parent_key).alias("_parent_key")),
        col(child_key) == col("_parent_key"),
        "left_anti"
    ).filter(col(child_key).isNotNull())
    
    orphaned_count = orphaned_df.count()
    passed = (orphaned_count == 0)
    pass_rate = 1.0 - (orphaned_count / total_rows) if total_rows > 0 else 1.0
    
    return ValidationResult(
        rule_name=rule_name,
        passed=passed,
        total_rows=total_rows,
        failed_rows=orphaned_count,
        pass_rate=pass_rate,
        message=f"Orphaned records: {orphaned_count:,} values in '{child_key}' not found in '{parent_key}'"
    )

In [0]:
def print_validation_results(results: List[ValidationResult]) -> None:
    """
    Print validation results in readable format.
    
    Args:
        results: List of ValidationResult objects
    
    Example:
        results = []
        results.extend(validate_not_null(df, ["id", "name"]))
        results.append(validate_unique(df, ["id"]))
        print_validation_results(results)
    """
    print("="*80)
    print("DATA QUALITY VALIDATION RESULTS".center(80))
    print("="*80)
    
    passed_count = sum(1 for r in results if r.passed)
    failed_count = len(results) - passed_count
    
    print(f"\nTotal Rules: {len(results)}")
    print(f"Passed: {passed_count}")
    print(f"Failed: {failed_count}")
    print()
    
    for result in results:
        status = "✓ PASS" if result.passed else "✗ FAIL"
        print(f"{status} | {result.rule_name}")
        print(f"       {result.message}")
        print(f"       Pass Rate: {result.pass_rate:.2%}")
        print()
    
    print("="*80)

def get_failed_validations(results: List[ValidationResult]) -> List[ValidationResult]:
    """
    Get only failed validation results.
    
    Args:
        results: List of ValidationResult objects
    
    Returns:
        List of failed ValidationResult objects
    
    Example:
        failed = get_failed_validations(results)
        if failed:
            raise ValueError(f"{len(failed)} data quality checks failed")
    """
    return [r for r in results if not r.passed]